# Chạy cross-validation trên Colab

Notebook này chạy mọi thứ trên Colab, laptop không chạy gì:
- mục 4: CV model thống kê và cây (tham số lấy từ `results/best_params.json` đã tune trước);
- mục 5–6: tune RNN/LSTM, rồi CV bản chính (5 seed) và bản `_ar` (3 seed);
- mục 8: dựng bảng và hình cho paper.

Mọi kết quả ghi vào Drive `MyDrive/rsf/results`. Xong thì tải thư mục đó về đặt vào repo (mục 9). Colab bản miễn phí chỉ có 2 lõi nên có thể cần vài phiên.

**Trước khi chạy:**
1. Trên laptop, `git push origin main` để Colab lấy được code mới nhất.
2. Trong Google Drive tạo thư mục `MyDrive/rsf/data/`, rồi upload 2 file từ `D:\retail-sales-forecasting-research\data\processed\`: `daily_series.csv` và `promotions.csv`.
3. Runtime → Change runtime type → **CPU**. GPU không giúp gì, vì code chạy torch trên CPU.

**Khi đang chạy:**
- Giữ tab mở. Colab bản miễn phí ngắt khi tab bỏ không lâu, và mỗi phiên tối đa khoảng 12 giờ.
- Bị ngắt thì chạy lại từ cell đầu. Mỗi task ghi một file vào Drive, nên task đã xong sẽ được bỏ qua.
- Không chạy stage `test` ở đây.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/rsf'
os.makedirs(f'{BASE}/results', exist_ok=True)
os.environ['RSF_PROCESSED_DIR'] = f'{BASE}/data'
os.environ['RSF_RESULTS_DIR'] = f'{BASE}/results'
os.environ['RSF_TORCH_THREADS'] = '1'   # Colab có 2 lõi, chạy --jobs 2 nên mỗi tiến trình 1 luồng
os.environ['PYTHONUTF8'] = '1'
print(os.listdir(f'{BASE}/data'))       # phải thấy daily_series.csv và promotions.csv
print('CPU:', os.cpu_count())

## 1. Lấy code

Repo private thì nhập token GitHub khi được hỏi. Token chỉ nhập qua ô `getpass`, đừng dán vào cell. Clone xong, token được xóa khỏi cấu hình git.

In [ ]:
import getpass, subprocess
REPO = 'github.com/myuyen0304/retail-sales-forecasting-research.git'
if not os.path.exists('/content/rsf'):
    token = getpass.getpass('GitHub token (để trống nếu repo public): ')
    url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
    subprocess.run(['git', 'clone', '--depth', '1', '-q', url, '/content/rsf'], check=True)
    subprocess.run(['git', '-C', '/content/rsf', 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)
    del token, url
%cd /content/rsf
!git log -1 --oneline

## 2. Cài thư viện và kiểm code

Colab có sẵn numpy, pandas và torch, nhưng phiên bản khác laptop. Code mới chỉ được kiểm trên laptop (pandas 3.1). Nếu pytest báo lỗi, **dừng lại** và gửi toàn bộ thông báo lỗi về để sửa, đừng chạy tiếp.

In [ ]:
!pip -q install -e . statsmodels lightgbm xgboost prophet pyarrow
import importlib.metadata as md
print({p: md.version(p) for p in ['pandas', 'numpy', 'joblib', 'torch', 'scikit-learn']})
!python -m pytest -q

## 3. Chép tham số đã tune từ repo sang Drive

Tham số của 7 model thống kê và cây đã tune trên laptop và nằm trong `results/` của repo. Bước này chép chúng sang Drive. Không chép thì lệnh tune sẽ tìm không thấy và có thể chạy lại từ đầu.

In [ ]:
import json, shutil
R = os.environ['RSF_RESULTS_DIR']
os.makedirs(f'{R}/tuning', exist_ok=True)
for f in os.listdir('results/tuning'):
    if not os.path.exists(f'{R}/tuning/{f}'):
        shutil.copy(f'results/tuning/{f}', f'{R}/tuning/{f}')
repo_bp = json.load(open('results/best_params.json'))
drive_bp = json.load(open(f'{R}/best_params.json')) if os.path.exists(f'{R}/best_params.json') else {}
json.dump({**repo_bp, **drive_bp}, open(f'{R}/best_params.json', 'w'), indent=2)
print(sorted(json.load(open(f'{R}/best_params.json'))))

## 4. CV model thống kê và cây (4 chuỗi × 2 target × 4 fold)

Dòng thứ hai là bản "pure" và SARIMA s=7, dùng cho ablation RQ2.

In [ ]:
!python -m rsf.run cv --models baselines,arima,sarima,sarimax,prophet,rf,gb,xgboost,lightgbm,ablation --jobs 2
!python -m rsf.run cv --models sarima_w7,prophet_pure,rf_lag,gb_lag,xgboost_lag,lightgbm_lag --jobs 2

## 5. Tune RNN/LSTM (TOTAL / revenue, 4 fold, 9 cấu hình mỗi model)

Phải chạy xong bước này rồi mới chạy mục 6. Nếu CV chạy trước, kết quả RNN/LSTM bị khóa ở tham số mặc định.

In [ ]:
!python -m rsf.run tune --models rnn,lstm --jobs 2
!cat $RSF_RESULTS_DIR/best_params.json

## 6. CV RNN/LSTM (4 chuỗi × 2 target × 4 fold)

- RNN/LSTM bản chính: 5 seed, 160 task mỗi model.
- Bản `_ar`: 3 seed, 96 task mỗi model.

Bước này lâu nhất. Mục 7 bên dưới cho biết đã chạy được bao nhiêu task và ước tính thời gian còn lại.

In [ ]:
_bp = json.load(open(f"{os.environ['RSF_RESULTS_DIR']}/best_params.json"))
assert 'rnn' in _bp and 'lstm' in _bp, 'Chưa tune xong RNN/LSTM: chạy lại mục 5 trước'
!python -m rsf.run cv --models rnn,lstm --jobs 2
!python -m rsf.run cv --models rnn_ar,lstm_ar --jobs 2 --seeds 3

## 7. Theo dõi tiến độ

Trong lúc một cell đang chạy thì không chạy được cell khác. Chạy cell này khi cell kia đã dừng hoặc bị ngắt. Nếu mở phiên mới, chạy lại các cell ở đầu notebook và mục 1–2 trước.

Model thống kê và cây cần 32 file mỗi model. Ước tính giờ còn lại chỉ tính cho RNN/LSTM.

In [ ]:
import pandas as pd
from pathlib import Path
R = Path(os.environ['RSF_RESULTS_DIR'])
P = R / 'preds/cv'
count = lambda m: len([p for p in (P / m).glob('*.parquet') if not p.stem.endswith('_fitted')])
print('Đã xong:', {d.name: count(d.name) for d in sorted(P.iterdir())} if P.exists() else {})
need = {'rnn': 160, 'lstm': 160, 'rnn_ar': 96, 'lstm_ar': 96}
done = {m: count(m) for m in need}
print('RNN/LSTM:', {m: f'{done[m]}/{need[m]}' for m in need})
if (R / 'runs_cv.csv').exists():
    log = pd.read_csv(R / 'runs_cv.csv')
    ok = log[log.status == 'ok']
    sec = ok[ok.model.isin(list(need))].groupby('model').fit_s.median()
    print('Số giây fit (trung vị):', sec.round(0).to_dict())
    left = sum((need[m] - done[m]) * sec.get(m, sec.max() if len(sec) else 0) for m in need) / 2 / 3600
    print(f'Ước tính còn khoảng {left:.1f} giờ (với --jobs 2)')
    print('Task lỗi:', (log.status != 'ok').sum())

## 8. Dựng bảng và hình

Chạy khi mục 7 báo đủ: mỗi model thống kê/cây 32 file, RNN/LSTM đủ số cần. Bảng ghi vào `results/tables`, hình vào `results/figures` trên Drive.

In [ ]:
!python scripts/make_tables.py
!python scripts/make_figures.py
print(sorted(os.listdir(f"{os.environ['RSF_RESULTS_DIR']}/tables")))

## 9. Đặt kết quả vào repo trên laptop (chỉ chép file, không chạy gì)

1. Trên Google Drive (web), chuột phải thư mục `MyDrive/rsf/results` → Download. Drive nén thành file zip, có thể chia nhiều file zip.
2. Cắt thư mục `D:\retail-sales-forecasting-research\results` cũ ra ngoài repo, ví dụ thành `D:\rsf_results_cu` (giữ làm bản sao, đừng xóa).
3. Giải nén file zip, đặt thư mục `results` vào `D:\retail-sales-forecasting-research\results`. Nhiều file zip thì giải nén hết vào cùng thư mục đó.
4. Báo Claude đọc `results/tables`.